In [31]:
import polars as pl

In [32]:
# 1. Загрузка датасета и первичный анализ
# Загрузка
data_set = pl.scan_parquet("data/NF-CSE-CIC-IDS2018-V2.parquet")

# Схема и список колонок
schema = data_set.collect_schema()
columns = schema.names()

# Число строк
total_rows = data_set.select(pl.len().alias("n_rows")).collect().item()

# Первичный анализ
print("\nПервичный анализ:")

# - форма датасета (количество строк и колонок)
print(f"1. Форма датасета: {total_rows:,} строк и {len(columns)} колонок")

# - типы колонок (последние пять)
last_five_col = columns[-5:]
print("\n2. Типы колонок (последние пять):")
for col_name in last_five_col:
    print(f"  {col_name}: {schema[col_name]}")

# - уникальные значения в колонке Label
unique_labels = (
    data_set
    .select(pl.col("Label").unique())
    .collect()
    .get_column("Label")
    .to_list()
)
print("\n3. Уникальные значения в колонке Label:")
print(unique_labels)

# - уникальные значения в колонке Attack
unique_attacks = (
    data_set
    .select(pl.col("Attack").unique())
    .collect()
    .get_column("Attack")
    .to_list()
)
print("\n4. Уникальные значения в колонке Attack:")
print(unique_attacks)


Первичный анализ:
1. Форма датасета: 17,129,715 строк и 43 колонок

2. Типы колонок (последние пять):
  DNS_QUERY_TYPE: Int16
  DNS_TTL_ANSWER: Int32
  FTP_COMMAND_RET_CODE: Int8
  Label: Int8
  Attack: String

3. Уникальные значения в колонке Label:
[0, 1]

4. Уникальные значения в колонке Attack:
['DoS attacks-GoldenEye', 'SQL Injection', 'Bot', 'Brute Force -Web', 'SSH-Bruteforce', 'DoS attacks-Slowloris', 'Infilteration', 'Benign', 'DoS attacks-SlowHTTPTest', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-Hulk', 'Brute Force -XSS', 'DDOS attack-LOIC-UDP', 'DDOS attack-HOIC', 'FTP-BruteForce']


In [33]:
# 2. Распределение по меткам
print("\nРаспределение по меткам:")
label_counts = (
    data_set
    .group_by("Label")
    .agg(pl.len().alias("count"))
    .sort("Label")
    .collect()
)

# Добавляем проценты
label_counts = label_counts.with_columns(
    (pl.col("count") * 100.0 / total_rows).alias("percent")
)

print(label_counts)


Распределение по меткам:
shape: (2, 3)
┌───────┬──────────┬───────────┐
│ Label ┆ count    ┆ percent   │
│ ---   ┆ ---      ┆ ---       │
│ i8    ┆ u32      ┆ f64       │
╞═══════╪══════════╪═══════════╡
│ 0     ┆ 15101685 ┆ 88.160749 │
│ 1     ┆ 2028030  ┆ 11.839251 │
└───────┴──────────┴───────────┘


In [34]:
# 3. Создание бинарного признака
print("\nСоздание бинарного признака is_attack")
df_with_flag = data_set.with_columns(
    pl.when(pl.col("Label") != 0)
    .then(pl.lit(1))
    .otherwise(pl.lit(0))
    .alias("is_attack")
)

# Предпросмотр (проверка)
print("Признак is_attack добавлен (5 строк):")
print(
    df_with_flag
    .select(["Label", "is_attack"])
    .head(5)
    .collect()
)


Создание бинарного признака is_attack
Признак is_attack добавлен (5 строк):
shape: (5, 2)
┌───────┬───────────┐
│ Label ┆ is_attack │
│ ---   ┆ ---       │
│ i8    ┆ i32       │
╞═══════╪═══════════╡
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
└───────┴───────────┘


In [35]:
# 4. Агрегация по типам атак 
# (только атаки Label != 0)
print("\nАгрегация по типам атак")
attack_agg = (
    df_with_flag
    .filter(pl.col("Label") != 0)
    .group_by("Attack")
    .agg(
        pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_flow_duration_ms"),
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.len().alias("record_count")
    )
    .sort("avg_in_bytes", descending=True)
    .collect()
)

print(attack_agg)


Агрегация по типам атак
shape: (14, 4)
┌──────────────────────────┬──────────────────────┬──────────────┬──────────────┐
│ Attack                   ┆ avg_flow_duration_ms ┆ avg_in_bytes ┆ record_count │
│ ---                      ┆ ---                  ┆ ---          ┆ ---          │
│ str                      ┆ f64                  ┆ f64          ┆ u32          │
╞══════════════════════════╪══════════════════════╪══════════════╪══════════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6             ┆ 5.8540e6     ┆ 2112         │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6             ┆ 25991.904925 ┆ 207078       │
│ Brute Force -XSS         ┆ 3.7568e6             ┆ 16871.281553 ┆ 927          │
│ Brute Force -Web         ┆ 3.7553e6             ┆ 9797.653756  ┆ 2143         │
│ SSH-Bruteforce           ┆ 733291.768981        ┆ 5828.140747  ┆ 94979        │
│ …                        ┆ …                    ┆ …            ┆ …            │
│ Infilteration            ┆ 234910.506514        ┆ 791.85

In [36]:
# 5. Топ-3 атак по трафику
print("\nТоп-3 атак по среднему объёму входящего трафика (avg_in_bytes)")
top3_attacks_by_inbytes = attack_agg.head(3)
print(top3_attacks_by_inbytes)


Топ-3 атак по среднему объёму входящего трафика (avg_in_bytes)
shape: (3, 4)
┌────────────────────────┬──────────────────────┬──────────────┬──────────────┐
│ Attack                 ┆ avg_flow_duration_ms ┆ avg_in_bytes ┆ record_count │
│ ---                    ┆ ---                  ┆ ---          ┆ ---          │
│ str                    ┆ f64                  ┆ f64          ┆ u32          │
╞════════════════════════╪══════════════════════╪══════════════╪══════════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6             ┆ 5.8540e6     ┆ 2112         │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6             ┆ 25991.904925 ┆ 207078       │
│ Brute Force -XSS       ┆ 3.7568e6             ┆ 16871.281553 ┆ 927          │
└────────────────────────┴──────────────────────┴──────────────┴──────────────┘


In [37]:
# 6. Распределение по протоколам
print("\nРаспределение по протоколам")

# - Общее распределение по PROTOCOL
print("\n1. Общее распределение по PROTOCOL")
proto_total = (
    df_with_flag
    .group_by("PROTOCOL")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") * 100.0 / total_rows).alias("percent")
    )
    .sort("count", descending=True)
    .collect()
)
print(proto_total)

# - Распределение по PROTOCOL только для Benign
print("\n2. Распределение по PROTOCOL только для Benign")
benign_count = label_counts.filter(pl.col("Label") == 0)["count"].item()
proto_benign = (
    df_with_flag
    .filter(pl.col("Label") == 0)
    .group_by("PROTOCOL")
    .agg(pl.len().alias("count_benign"))
    .with_columns(
        (pl.col("count_benign") * 100.0 / benign_count).alias("percent_benign")
        if benign_count > 0 else pl.lit(0.0).alias("percent_benign")
    )
    .sort("count_benign", descending=True)
    .collect()
)
print(proto_benign)

# - Распределение по PROTOCOL только для Attack (с разбивкой по Attack)
print("\n3. Распределение по PROTOCOL только для Attack (разбивка по Attack)")
attack_count = label_counts.filter(pl.col("Label") == 1)["count"].item()
proto_attack_by_attack = (
    df_with_flag
    .filter(pl.col("Label") != 0)
    .group_by(["Attack", "PROTOCOL"])
    .agg(pl.len().alias("count_attack"))
    .with_columns(
        (pl.col("count_attack") * 100.0 / attack_count).alias("percent_attack_overall")
        if attack_count > 0 else pl.lit(0.0).alias("percent_attack_overall")
    )
    .sort(["Attack", "count_attack"], descending=[False, True])
    .collect()
)
print(proto_attack_by_attack)

# - Сравнение PROTOCOL между Benign и Attack
print("\n4. Сравнение PROTOCOL между Benign и Attack")
proto_attack_total = (
    df_with_flag
    .filter(pl.col("Label") != 0)
    .group_by("PROTOCOL")
    .agg(pl.len().alias("count_attack"))
    .with_columns(
        (pl.col("count_attack") * 100.0 / attack_count).alias("percent_attack")
        if attack_count > 0 else pl.lit(0.0).alias("percent_attack")
    )
    .sort("count_attack", descending=True)
    .collect()
)

proto_comparison = (
    proto_benign
    .select(["PROTOCOL", "count_benign", "percent_benign"])
    .join(
        proto_attack_total.select(["PROTOCOL", "count_attack", "percent_attack"]),
        on="PROTOCOL",
        how="full",
        coalesce=True
    )
    .fill_null(0)
    .sort("count_attack", descending=True)
)
print(proto_comparison)


Распределение по протоколам

1. Общее распределение по PROTOCOL
shape: (6, 3)
┌──────────┬─────────┬───────────┐
│ PROTOCOL ┆ count   ┆ percent   │
│ ---      ┆ ---     ┆ ---       │
│ i8       ┆ u32     ┆ f64       │
╞══════════╪═════════╪═══════════╡
│ 6        ┆ 9346287 ┆ 54.561836 │
│ 17       ┆ 7776756 ┆ 45.399214 │
│ 1        ┆ 4857    ┆ 0.028354  │
│ 2        ┆ 976     ┆ 0.005698  │
│ 58       ┆ 836     ┆ 0.00488   │
│ 47       ┆ 3       ┆ 0.000018  │
└──────────┴─────────┴───────────┘

2. Распределение по PROTOCOL только для Benign
shape: (6, 3)
┌──────────┬──────────────┬────────────────┐
│ PROTOCOL ┆ count_benign ┆ percent_benign │
│ ---      ┆ ---          ┆ ---            │
│ i8       ┆ u32          ┆ f64            │
╞══════════╪══════════════╪════════════════╡
│ 17       ┆ 7688529      ┆ 50.911729      │
│ 6        ┆ 7406897      ┆ 49.046825      │
│ 1        ┆ 4537         ┆ 0.030043       │
│ 2        ┆ 883          ┆ 0.005847       │
│ 58       ┆ 836          ┆ 0.0055

In [38]:
# 7. Сравнение метрик: Benign vs Attack
print("\nСравнение метрик: Benign (0) vs Attack (1) по is_attack")
metrics_by_is_attack = (
    df_with_flag
    .group_by("is_attack")
    .agg(
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.col("OUT_BYTES").mean().alias("avg_out_bytes"),
        pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_flow_duration_ms"),
        pl.len().alias("record_count")
    )
    .sort("is_attack")
    .collect()
)
print(metrics_by_is_attack)


Сравнение метрик: Benign (0) vs Attack (1) по is_attack
shape: (2, 5)
┌───────────┬──────────────┬───────────────┬──────────────────────┬──────────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_flow_duration_ms ┆ record_count │
│ ---       ┆ ---          ┆ ---           ┆ ---                  ┆ ---          │
│ i32       ┆ f64          ┆ f64           ┆ f64                  ┆ u32          │
╞═══════════╪══════════════╪═══════════════╪══════════════════════╪══════════════╡
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107        ┆ 15101685     │
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6             ┆ 2028030      │
└───────────┴──────────────┴───────────────┴──────────────────────┴──────────────┘


In [39]:
# 8. (Бонус) Эвристика детектирования
print("\nЭвристика детектирования is_suspicious")

df_with_heuristic = (
    df_with_flag
    .with_columns(
        (pl.col("IN_BYTES") / (pl.col("OUT_BYTES") + 1)).alias("bytes_ratio"),
        (pl.col("IN_BYTES") + pl.col("OUT_BYTES")).alias("total_bytes"),
        (
            (pl.col("IN_BYTES") + pl.col("OUT_BYTES")) /
            (pl.col("IN_PKTS") + pl.col("OUT_PKTS") + 1)
        ).alias("packet_size_avg")
    )
    .with_columns(
        pl.when(
            (pl.col("bytes_ratio") > 10) &
            (pl.col("FLOW_DURATION_MILLISECONDS") < 500) &
            (pl.col("IN_PKTS") > 10)
        )
        .then(pl.lit(1))
        .otherwise(pl.lit(0))
        .alias("is_suspicious")
    )
)

# Расчеты по эвристике
heuristic_stats = (
    df_with_heuristic
    .group_by("is_suspicious")
    .agg(
        pl.len().alias("count_flagged"),
        # Сколько из помеченных реально являются атаками (Label != 0)
        (pl.col("Label") != 0).sum().alias("true_attacks_count")
    )
    .collect()
)

print("Статистика эвристики is_suspicious:")
print(heuristic_stats)

# Определение точности эвристики: true_attacks / total_flagged
row_flagged = heuristic_stats.filter(pl.col("is_suspicious") == 1)
if row_flagged.height > 0:
    total_flagged = row_flagged["count_flagged"].item()
    true_attacks_flagged = row_flagged["true_attacks_count"].item()
    heuristic_precision = (
        true_attacks_flagged / total_flagged if total_flagged > 0 else 0.0
    )
else:
    total_flagged = 0
    true_attacks_flagged = 0
    heuristic_precision = 0.0

print("\nРезультаты эвристики:")
print(f"  Всего записей помечено как is_suspicious == 1: {total_flagged:,}")
print(f"  Из них на самом деле являются атаками (Label != 0): {true_attacks_flagged:,}")
print(f"  Точность эвристики (true_attacks / total_flagged): {heuristic_precision:.4f}")

# Сохранение итоговой агрегации (по типам атак) в файл attack_summary_by_type.parquet
attack_agg.write_parquet("data/attack_summary_by_type.parquet")
print(f"\nИтоговая агрегация (по типам атак) сохранена в файл: data/attack_summary_by_type.parquet")


Эвристика детектирования is_suspicious
Статистика эвристики is_suspicious:
shape: (2, 3)
┌───────────────┬───────────────┬────────────────────┐
│ is_suspicious ┆ count_flagged ┆ true_attacks_count │
│ ---           ┆ ---           ┆ ---                │
│ i32           ┆ u32           ┆ u32                │
╞═══════════════╪═══════════════╪════════════════════╡
│ 1             ┆ 4645          ┆ 3707               │
│ 0             ┆ 17125070      ┆ 2024323            │
└───────────────┴───────────────┴────────────────────┘

Результаты эвристики:
  Всего записей помечено как is_suspicious == 1: 4,645
  Из них на самом деле являются атаками (Label != 0): 3,707
  Точность эвристики (true_attacks / total_flagged): 0.7981

Итоговая агрегация (по типам атак) сохранена в файл: data/attack_summary_by_type.parquet
